# E4.4: XLM-R large emotion on a Colab GPU

Runs the pre-registered config `configs/E4_4_xlmr_large_emotion.json` (3 seeds, about 30 to 45 min each on a T4) and saves runs to Google Drive, so a disconnect loses at most one seed. Re-running the run cell skips finished seeds.

**Before you start:** Runtime > Change runtime type > T4 GPU. Then Runtime > Run all.

If `freeze_splits.py --check` fails, stop: the data split differs from the one every other model used.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/mhrisk'
os.makedirs(BASE, exist_ok=True)
os.environ['MHRISK_RUNS_DIR'] = f'{BASE}/runs'
os.environ['HF_HOME'] = f'{BASE}/hf_cache'

In [ ]:
BRANCH = 'paper-v2-probe'
import os
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} https://github.com/faizdevx/Multilingual-Mental-Health-Risk-.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r requirements-colab.txt

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py > /dev/null
!python scripts/freeze_splits.py --check

In [ ]:
!python scripts/run_experiment.py configs/E4_4_xlmr_large_emotion.json --device cuda

## Download the results (metrics and per-example probabilities only; no text, no checkpoints)

In [ ]:
!cd {BASE} && zip -qr e4_4_results.zip runs/E4_4_xlmr_large_emotion -x '*/resume*' && ls -lh e4_4_results.zip
from google.colab import files
files.download(f'{BASE}/e4_4_results.zip')